# Simple Python Notebook

A basic Jupyter notebook demonstrating Python code execution.

In [ ]:
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# Global storage for layer data
layer_data = {}  # {layer_idx: {'neurons': {neuron_idx: {...}}, 'plot_output': widget}}

def compute_neuron_output(weights, bias, x_values, activation='relu'):
    """Compute neuron output: activation(w*x + b)"""
    z = np.zeros_like(x_values)
    for w in weights:
        z += w * x_values
    z += bias
    
    # Apply activation function
    if activation == 'relu':
        return np.maximum(0, z)
    elif activation == 'linear':
        return z
    elif activation == 'sigmoid':
        return 1 / (1 + np.exp(-z))
    return z

def compute_layer_outputs(layer_idx, x):
    """Compute all neuron outputs for a specific layer"""
    if layer_idx not in layer_data or 'neurons' not in layer_data[layer_idx]:
        return []
    
    neurons = layer_data[layer_idx]['neurons']
    outputs = []
    
    for neuron_idx in sorted(neurons.keys()):
        neuron_info = neurons[neuron_idx]
        sliders = neuron_info['sliders']
        weights = [s.value for s in sliders[:-1]]  # All except bias
        bias = sliders[-1].value  # Last slider is bias
        
        # For layer 0, input is x directly
        if layer_idx == 0:
            y = compute_neuron_output(weights, bias, x, activation='relu')
        else:
            # For other layers, compute weighted sum of previous layer outputs
            prev_outputs = compute_layer_outputs(layer_idx - 1, x)
            if len(prev_outputs) == 0:
                y = np.zeros_like(x)
            else:
                # Compute: activation(w1*prev_out1 + w2*prev_out2 + ... + b)
                z = np.zeros_like(x)
                for i, (w, prev_out) in enumerate(zip(weights, prev_outputs)):
                    z += w * prev_out
                z += bias
                y = np.maximum(0, z)  # ReLU activation
        
        outputs.append(y)
    
    return outputs

def update_layer_plot(layer_idx):
    """Update plot for a specific layer showing its checked neurons"""
    if layer_idx not in layer_data:
        return
    
    plot_output = layer_data[layer_idx]['plot_output']
    neurons = layer_data[layer_idx]['neurons']
    
    with plot_output:
        clear_output(wait=True)
        
        fig, ax = plt.subplots(figsize=(8, 5))
        x = np.linspace(-10, 10, 1000)
        
        colors = plt.cm.tab10(np.linspace(0, 1, 10))
        plot_count = 0
        
        # Compute all outputs for this layer
        layer_outputs = compute_layer_outputs(layer_idx, x)
        
        for neuron_idx in sorted(neurons.keys()):
            neuron_info = neurons[neuron_idx]
            checkbox = neuron_info['checkbox']
            
            if checkbox.value:  # Only plot if checked
                if neuron_idx < len(layer_outputs):
                    y = layer_outputs[neuron_idx]
                    
                    label = f'Neuron {neuron_idx+1}'
                    ax.plot(x, y, label=label, color=colors[neuron_idx % 10], linewidth=2)
                    plot_count += 1
        
        if plot_count > 0:
            ax.set_xlabel('Input (x)', fontsize=11)
            ax.set_ylabel('Output', fontsize=11)
            title = f'Layer {layer_idx + 1} - Neuron Outputs'
            if layer_idx > 0:
                title += ' (using Layer {} outputs)'.format(layer_idx)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.legend(loc='best', fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.set_ylim(-5, 15)
        else:
            ax.text(0.5, 0.5, 'No neurons selected\nCheck boxes to plot', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=12, color='gray')
            ax.set_xlim(-10, 10)
            ax.set_ylim(-5, 15)
        
        plt.tight_layout()
        plt.show()

def update_all_downstream_plots(layer_idx):
    """Update this layer's plot and all downstream layers"""
    # Update current layer
    update_layer_plot(layer_idx)
    
    # Update all layers that depend on this one
    max_layer = max(layer_data.keys()) if layer_data else 0
    for next_layer in range(layer_idx + 1, max_layer + 1):
        update_layer_plot(next_layer)

def single_neuron(layer_idx, neuron_idx, parameters):
    """Create a single neuron with sliders and checkbox"""
    slider_list = []
    
    # Weight sliders
    for i in range(parameters):
        slider = widgets.FloatSlider(
            value=0, min=-3, max=3, step=0.1, 
            orientation='vertical', description=f"w{i+1}"
        )
        slider.observe(lambda change, l=layer_idx: update_all_downstream_plots(l), names='value')
        slider_list.append(slider)
    
    # Bias slider
    bias_slider = widgets.FloatSlider(
        value=0, min=-3, max=3, step=0.1, 
        orientation='vertical', description="b"
    )
    bias_slider.observe(lambda change, l=layer_idx: update_all_downstream_plots(l), names='value')
    slider_list.append(bias_slider)
    
    # Checkbox for plot visibility
    checkbox = widgets.Checkbox(
        value=False,
        description='Plot',
        indent=False
    )
    checkbox.observe(lambda change, l=layer_idx: update_layer_plot(l), names='value')
    
    # Store neuron data
    if layer_idx not in layer_data:
        layer_data[layer_idx] = {'neurons': {}, 'plot_output': widgets.Output()}
    
    layer_data[layer_idx]['neurons'][neuron_idx] = {
        'checkbox': checkbox,
        'sliders': slider_list
    }
    
    return widgets.VBox([
        checkbox,
        widgets.HBox(slider_list)
    ])

def single_layer(layer_idx, neurons, parameters):
    """Create a layer with multiple neurons"""
    neuron_list = []
    for i in range(neurons):
        neuron_label = widgets.HTML(value=f"<b>Neuron {i+1}</b>")
        neuron_widget = single_neuron(layer_idx, i, parameters)
        neuron_list.append(widgets.VBox([neuron_label, neuron_widget]))
    
    neurons_box = widgets.HBox(neuron_list)
    
    # Get or create plot output for this layer
    if layer_idx not in layer_data:
        layer_data[layer_idx] = {'neurons': {}, 'plot_output': widgets.Output()}
    
    plot_output = layer_data[layer_idx]['plot_output']
    
    # Return neurons and plot side by side
    return widgets.HBox([neurons_box, plot_output])

def neural_network(layers):
    """Creates the neural network UI with dynamic layers"""
    clear_output(wait=True)
    
    # Clear previous layer data
    layer_data.clear()
    
    all_layers = []
    neuron_slider_list = []
    update_functions = []
    
    for layer_idx in range(layers):
        # Last layer always has 1 neuron (output)
        if layer_idx == layers - 1:
            neuron_slider = widgets.IntSlider(value=1, min=1, max=1, step=1, description='Neurons:', disabled=True)
        else:
            neuron_slider = widgets.IntSlider(value=1, min=1, max=10, step=1, description='Neurons:')
        neuron_slider_list.append(neuron_slider)
        
        layer_output = widgets.Output()
        
        def update_layer(change, output=layer_output, n_slider=neuron_slider, idx=layer_idx):
            with output:
                clear_output(wait=True)
                # First layer: 1 input weight
                # Other layers: weights = neurons in previous layer
                if idx == 0:
                    num_weights = 1
                else:
                    num_weights = neuron_slider_list[idx - 1].value
                layer_widget = single_layer(idx, n_slider.value, num_weights)
                display(layer_widget)
            update_all_downstream_plots(idx)  # Refresh plot when layer changes
        
        update_functions.append(update_layer)
        neuron_slider.observe(update_layer, names='value')
        
        # Initial display
        update_layer(None)
        
        layer_header = widgets.HTML(value=f"<h4>Layer {layer_idx + 1}</h4>")
        controls = widgets.HBox([neuron_slider])
        
        layer_box = widgets.VBox([
            layer_header,
            controls,
            layer_output
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px 0'))
        
        all_layers.append(layer_box)
    
    # Set up cross-layer observations
    for layer_idx in range(layers - 1):
        next_layer_update = update_functions[layer_idx + 1]
        neuron_slider_list[layer_idx].observe(next_layer_update, names='value')
    
    # Display network
    display(widgets.VBox(all_layers))

# Create UI
ui = widgets.VBox([
    widgets.HTML(value="<h2>Neural Network Configuration</h2>"),
    widgets.IntSlider(value=2, min=1, max=10, step=1, description='Layers:')
])
layer_slider = ui.children[1]
layers_output = widgets.interactive_output(neural_network, {'layers': layer_slider})
ui.children = ui.children + (layers_output,)
display(ui)

In [ ]:
%matplotlib inline
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

def f(m, b):
    plt.figure(2)
    x = np.linspace(-10, 10, num=1000)
    plt.plot(x, m * x + b)
    plt.plot(x, b * x + m)
    plt.ylim(-5, 5)
    plt.show()

# Create slider widgets first
m_slider = widgets.FloatSlider(value=0, min=-2.0, max=2.0, step=0.1, description='m:')
b_slider = widgets.FloatSlider(value=0, min=-3, max=3, step=0.5, description='b:')

# Pass widgets to interactive_output
output = widgets.interactive_output(f, {'m': m_slider, 'b': b_slider})

# Display sliders and output
display(widgets.HBox([m_slider, b_slider, output]))

3

30